In [1]:
# 05_sensitivity.ipynb
# Cross-Domain AI Governance Gap Analysis
# 05. Sensitivity Analysis for the noise-corrected GGI
#
# Test whether the two headline findings survive perturbations of the GGI design choices:
#   (F1) insurance shows a larger governance gap than credit
#   (F2) the regulation ordering of the GGI
# Studies: A threshold scaling, B aggregation rule, C explainability-threshold sweep,
#          D bootstrap confidence intervals.
# All component scores use the SAME significance-gated worst-case fairness as notebook 04:
# the worst case is taken only over sensitive attributes flagged significant there
# (results/tables/04_significant_attrs.json); if none, the fairness axes score parity (1.0).

import os
import json
import warnings
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score
from xgboost import XGBClassifier
import shap

warnings.filterwarnings("ignore", category=UserWarning)
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 150)

DATA_DIR = os.path.join("..", "data")
PROCESSED_DIR = os.path.join(DATA_DIR, "processed")
RESULTS_DIR = os.path.join("..", "results")
TAB_DIR = os.path.join(RESULTS_DIR, "tables")
os.makedirs(TAB_DIR, exist_ok=True)

RANDOM_STATE = 42
SHAP_SAMPLE = 1000
N_BOOTSTRAP = 200

BASE_REGULATIONS = {
    "KR_AI_Basic_Act": {"s_PERF": 0.30, "s_DI": 0.80, "s_SP": 0.90, "s_EXPL": 0.50},
    "EU_AI_Act":       {"s_PERF": 0.40, "s_DI": 0.85, "s_SP": 0.92, "s_EXPL": 0.60},
    "NAIC":            {"s_PERF": 0.40, "s_DI": 0.90, "s_SP": 0.95, "s_EXPL": 0.55},
}
DATASETS = [
    ("German Credit",        "german"),
    ("Credit Card Approval", "credit_card_approval"),
    ("Medical Cost",         "insurance"),
    ("Porto Seguro",         "porto_seguro"),
]
DOMAIN_OF = {}

with open(os.path.join(PROCESSED_DIR, "preprocess_meta.json")) as f:
    META = json.load(f)
with open(os.path.join(TAB_DIR, "04_significant_attrs.json")) as f:
    SIG_ATTRS = json.load(f)   # dataset key -> list of BH-significant sensitive columns

def load_split(key):
    return (pd.read_parquet(os.path.join(PROCESSED_DIR, f"{key}_train.parquet")),
            pd.read_parquet(os.path.join(PROCESSED_DIR, f"{key}_test.parquet")))

def get_Xy(df, key):
    meta = META[key]
    target_col = meta["target_col"]
    sensitive_cols = meta["sensitive_cols"]
    aux_cols = [c for c in ["target_60dpd", "target_30dpd"] if c in df.columns and c != target_col]
    drop = [target_col] + sensitive_cols + aux_cols
    X = df.drop(columns=[c for c in drop if c in df.columns], errors="ignore")
    return X, df[target_col].astype(int)

def group_di_spd(y_pred, pm, um):
    pp = y_pred[pm].mean() if pm.sum() else np.nan
    pu = y_pred[um].mean() if um.sum() else np.nan
    if not (np.isfinite(pp) and np.isfinite(pu)) or pp == 0:
        return np.nan, np.nan
    return pu / pp, pu - pp

# Significance-gated worst-case fairness: worst case over the pre-identified significant
# attributes only (from notebook 04). Empty significant set -> parity (no evidence).
def gated_worst_case(test_df, sig_cols, y_pred):
    dis, spds = [], []
    for col in sig_cols:
        if col not in test_df.columns:
            continue
        s = test_df[col]
        if s.nunique(dropna=True) < 2:
            continue
        priv = s.mode(dropna=True).iloc[0]
        pm = (s == priv).values
        um = (s != priv).values & s.notna().values
        di, spd = group_di_spd(y_pred, pm, um)
        if np.isfinite(di): dis.append(di)
        if np.isfinite(spd): spds.append(spd)
    if not dis and not spds:
        return 1.0, 1.0
    s_DI = min(1.0, min(dis)) if dis else 1.0
    s_SP = 1.0 - max(abs(x) for x in spds) if spds else 1.0
    return s_DI, s_SP

def shap_explainability(model, X, sample=SHAP_SAMPLE, seed=RANDOM_STATE):
    if len(X) > sample:
        X = X.sample(sample, random_state=seed)
    sv = shap.TreeExplainer(model).shap_values(X)
    if isinstance(sv, list):
        sv = sv[-1]
    mean_abs = np.abs(sv).mean(axis=0)
    total = mean_abs.sum()
    if total == 0 or len(mean_abs) <= 1:
        return 0.0
    p = mean_abs / total
    p = p[p > 0]
    return float(1.0 - (-np.sum(p * np.log(p))) / np.log(len(mean_abs)))

def component_gap(score, tau):
    if not np.isfinite(score):
        return np.nan
    return max(0.0, tau - score) / tau

def aggregate(gaps, rule="euclid"):
    gaps = [g for g in gaps if np.isfinite(g)]
    if not gaps:
        return np.nan
    if rule == "euclid":
        return float(np.sqrt(np.sum(np.square(gaps))))
    if rule == "mean":
        return float(np.mean(gaps))
    if rule == "max":
        return float(np.max(gaps))
    raise ValueError(rule)

def compute_ggi(scores, regulations, rule="euclid"):
    rows = []
    for name, sc in scores.items():
        for reg, tau in regulations.items():
            gaps = [component_gap(sc["s_PERF"], tau["s_PERF"]),
                    component_gap(sc["s_DI"], tau["s_DI"]),
                    component_gap(sc["s_SP"], tau["s_SP"]),
                    component_gap(sc["s_EXPL"], tau["s_EXPL"])]
            rows.append({"domain": sc["domain"], "dataset": name,
                         "regulation": reg, "GGI": aggregate(gaps, rule)})
    return pd.DataFrame(rows)

# Recompute base (noise-corrected) component scores; keep fitted artefacts for the bootstrap.
component_scores, fitted = {}, {}
for display_name, key in DATASETS:
    train, test = load_split(key)
    X_train, y_train = get_Xy(train, key)
    X_test, y_test = get_Xy(test, key)
    X_test = X_test[X_train.columns]
    n_pos, n_neg = int(y_train.sum()), int((y_train == 0).sum())
    spw = (n_neg / n_pos) if n_pos > 0 else 1.0
    model = XGBClassifier(n_estimators=300, max_depth=5, learning_rate=0.05,
        subsample=0.9, colsample_bytree=0.9, scale_pos_weight=spw,
        eval_metric="logloss", random_state=RANDOM_STATE, n_jobs=-1)
    model.fit(X_train, y_train)
    proba = model.predict_proba(X_test)[:, 1]
    thr = np.quantile(proba, 1.0 - float(y_train.mean()))
    y_pred = (proba >= thr).astype(int)
    auc = roc_auc_score(y_test.values, proba)
    s_PERF = max(0.0, (auc - 0.5) / 0.5)
    sig_cols = SIG_ATTRS.get(key, [])
    s_DI, s_SP = gated_worst_case(test, sig_cols, y_pred)
    s_EXPL = shap_explainability(model, X_test)
    domain = META[key].get("domain", "")
    DOMAIN_OF[display_name] = domain
    component_scores[display_name] = {"domain": domain, "s_PERF": s_PERF,
                                      "s_DI": s_DI, "s_SP": s_SP, "s_EXPL": s_EXPL}
    fitted[display_name] = {"key": key, "test": test, "proba": proba, "threshold": thr,
                            "s_EXPL": s_EXPL, "y_test": y_test, "sig_cols": sig_cols}
    print(f"[{display_name:>22}] base corrected scores captured")

def insurance_gt_credit(ggi_table):
    m = ggi_table.groupby("domain")["GGI"].mean()
    if "insurance" in m and "credit" in m:
        return bool(m["insurance"] > m["credit"]), float(m["insurance"] - m["credit"])
    return np.nan, np.nan

# Study A: threshold scaling
factors = [0.85, 0.90, 0.95, 1.00, 1.05, 1.10, 1.15]
studyA = []
for f in factors:
    scaled = {reg: {k: min(v * f, 0.999) for k, v in tau.items()}
              for reg, tau in BASE_REGULATIONS.items()}
    ggi = compute_ggi(component_scores, scaled)
    holds, gap = insurance_gt_credit(ggi)
    dmean = ggi.groupby("domain")["GGI"].mean()
    studyA.append({"factor": f, "credit_mean": round(dmean.get("credit", np.nan), 4),
                   "insurance_mean": round(dmean.get("insurance", np.nan), 4),
                   "insurance_gt_credit": holds, "gap": round(gap, 4)})
studyA_df = pd.DataFrame(studyA)
print("Study A - threshold scaling")
print(studyA_df.to_string(index=False))
studyA_df.to_csv(os.path.join(TAB_DIR, "05_sensitivity_threshold.csv"), index=False)

# Study B: aggregation rule
studyB = []
for rule in ["euclid", "mean", "max"]:
    ggi = compute_ggi(component_scores, BASE_REGULATIONS, rule=rule)
    holds, gap = insurance_gt_credit(ggi)
    reg_order = (ggi.groupby("regulation")["GGI"].mean()
                 .reindex(["KR_AI_Basic_Act", "EU_AI_Act", "NAIC"]).round(4))
    studyB.append({"rule": rule, "insurance_gt_credit": holds, "gap": round(gap, 4),
                   "KR": reg_order["KR_AI_Basic_Act"], "EU": reg_order["EU_AI_Act"],
                   "NAIC": reg_order["NAIC"]})
studyB_df = pd.DataFrame(studyB)
print("Study B - aggregation rule")
print(studyB_df.to_string(index=False))
studyB_df.to_csv(os.path.join(TAB_DIR, "05_sensitivity_aggregation.csv"), index=False)

# Study C: explainability-threshold sweep
expl_taus = [0.20, 0.30, 0.40, 0.50, 0.60, 0.70]
studyC = []
for et in expl_taus:
    reg = {r: {**tau, "s_EXPL": et} for r, tau in BASE_REGULATIONS.items()}
    ggi = compute_ggi(component_scores, reg)
    dmean = ggi.groupby("domain")["GGI"].mean()
    holds, gap = insurance_gt_credit(ggi)
    studyC.append({"tau_EXPL": et, "credit_mean": round(dmean.get("credit", np.nan), 4),
                   "insurance_mean": round(dmean.get("insurance", np.nan), 4),
                   "insurance_gt_credit": holds, "gap": round(gap, 4)})
studyC_df = pd.DataFrame(studyC)
print("Study C - explainability threshold sweep")
print(studyC_df.to_string(index=False))
studyC_df.to_csv(os.path.join(TAB_DIR, "05_sensitivity_explainability.csv"), index=False)

# Study D: bootstrap 95% CI (gated worst-case over the fixed significant set per resample)
def bootstrap_ggi(dataset_name, n_boot=N_BOOTSTRAP, seed=RANDOM_STATE):
    info = fitted[dataset_name]
    test = info["test"].reset_index(drop=True)
    proba = info["proba"]
    thr = info["threshold"]
    s_EXPL = info["s_EXPL"]
    y_test = info["y_test"].reset_index(drop=True).values
    sig_cols = info["sig_cols"]
    rng = np.random.default_rng(seed)
    n = len(test)
    ggis = {"KR_AI_Basic_Act": [], "EU_AI_Act": [], "NAIC": []}
    for _ in range(n_boot):
        idx = rng.integers(0, n, size=n)
        yb, pb = y_test[idx], proba[idx]
        if len(np.unique(yb)) < 2:
            continue
        yhat = (pb >= thr).astype(int)
        auc = roc_auc_score(yb, pb)
        s_PERF = max(0.0, (auc - 0.5) / 0.5)
        s_DI, s_SP = gated_worst_case(test.iloc[idx], sig_cols, yhat)
        sc = {dataset_name: {"domain": DOMAIN_OF[dataset_name], "s_PERF": s_PERF,
                             "s_DI": s_DI, "s_SP": s_SP, "s_EXPL": s_EXPL}}
        g = compute_ggi(sc, BASE_REGULATIONS)
        for reg in ggis:
            v = g.loc[g["regulation"] == reg, "GGI"].iloc[0]
            if np.isfinite(v):
                ggis[reg].append(v)
    return ggis

studyD = []
for display_name, _ in DATASETS:
    ci = bootstrap_ggi(display_name)
    for reg, vals in ci.items():
        arr = np.array(vals)
        studyD.append({"dataset": display_name, "domain": DOMAIN_OF[display_name],
                       "regulation": reg, "GGI_mean": round(arr.mean(), 4),
                       "CI_low": round(np.percentile(arr, 2.5), 4),
                       "CI_high": round(np.percentile(arr, 97.5), 4), "n_valid": len(arr)})
studyD_df = pd.DataFrame(studyD)
print("Study D - bootstrap 95% CI")
print(studyD_df.to_string(index=False))
studyD_df.to_csv(os.path.join(TAB_DIR, "05_sensitivity_bootstrap.csv"), index=False)

print("=== Robustness summary ===")
print("F1 holds across threshold factors:", studyA_df["insurance_gt_credit"].all())
print("F1 holds across aggregation rules:", studyB_df["insurance_gt_credit"].all())
print("F1 holds across explainability thresholds:", studyC_df["insurance_gt_credit"].all())
print("05_sensitivity complete")


[         German Credit] base corrected scores captured


[  Credit Card Approval] base corrected scores captured


[          Medical Cost] base corrected scores captured


[          Porto Seguro] base corrected scores captured
Study A - threshold scaling
 factor  credit_mean  insurance_mean  insurance_gt_credit    gap
   0.85       0.5994          0.9913                 True 0.3919
   0.90       0.6217          1.0215                 True 0.3998
   0.95       0.6419          1.0489                 True 0.4070
   1.00       0.6613          1.0737                 True 0.4124
   1.05       0.6800          1.0965                 True 0.4165
   1.10       0.6976          1.1173                 True 0.4197
   1.15       0.7142          1.1353                 True 0.4211
Study B - aggregation rule
  rule  insurance_gt_credit    gap     KR     EU   NAIC
euclid                 True 0.4124 0.8184 0.9006 0.8835
  mean                 True 0.2482 0.2746 0.3085 0.3046
   max                 True 0.1496 0.7093 0.7578 0.7358
Study C - explainability threshold sweep
 tau_EXPL  credit_mean  insurance_mean  insurance_gt_credit    gap
      0.2       0.1369          0.854

Study D - bootstrap 95% CI
             dataset    domain      regulation  GGI_mean  CI_low  CI_high  n_valid
       German Credit    credit KR_AI_Basic_Act    0.6874  0.6874   0.6874      200
       German Credit    credit       EU_AI_Act    0.7395  0.7395   0.7395      200
       German Credit    credit            NAIC    0.7158  0.7158   0.7158      200
Credit Card Approval    credit KR_AI_Basic_Act    0.5676  0.5676   0.5676      200
Credit Card Approval    credit       EU_AI_Act    0.6459  0.6397   0.6610      200
Credit Card Approval    credit            NAIC    0.6135  0.6069   0.6294      200
        Medical Cost insurance KR_AI_Basic_Act    0.9174  0.7313   1.2052      200
        Medical Cost insurance       EU_AI_Act    1.0153  0.8348   1.2407      200
        Medical Cost insurance            NAIC    0.9994  0.8156   1.2278      200
        Porto Seguro insurance KR_AI_Basic_Act    1.1331  1.0375   1.2314      200
        Porto Seguro insurance       EU_AI_Act    1.2204  1.